# 04 — Exploratory Data Analysis with Pandas

Use grouped tables and descriptive statistics. This project deliberately avoids plotting libraries.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'sales_cleaned.csv', parse_dates=['Order_Date'])
df.head()

## Create derived columns

In [ ]:
df['Gross_Sales'] = df['Quantity'] * df['Unit_Price']
df['Net_Sales'] = df['Gross_Sales'] * (1 - df['Discount'])
df['Month'] = df['Order_Date'].dt.to_period('M').astype('string')
df['Weekday'] = df['Order_Date'].dt.day_name()
print(df[['Gross_Sales', 'Net_Sales']].describe())

## Product and regional performance

In [ ]:
product_summary = df.groupby('Product', as_index=False).agg(
    Units_Sold=('Quantity', 'sum'),
    Orders=('Order_ID', 'nunique'),
    Revenue=('Net_Sales', 'sum'),
    Avg_Order_Line=('Net_Sales', 'mean')
).sort_values('Revenue', ascending=False)
product_summary

In [ ]:
region_summary = df.groupby('Region', as_index=False).agg(
    Revenue=('Net_Sales', 'sum'),
    Customers=('Customer_ID', 'nunique'),
    Orders=('Order_ID', 'nunique')
).sort_values('Revenue', ascending=False)
region_summary

## Monthly sales and growth

In [ ]:
monthly = df.groupby('Month', as_index=False).agg(Revenue=('Net_Sales', 'sum'), Orders=('Order_ID', 'nunique')).sort_values('Month')
monthly['Previous_Month_Revenue'] = monthly['Revenue'].shift(1)
monthly['Growth_Rate'] = (monthly['Revenue'] / monthly['Previous_Month_Revenue']) - 1
monthly

## Customer summaries

In [ ]:
customer_summary = df.groupby('Customer_ID', as_index=False).agg(
    Orders=('Order_ID', 'nunique'),
    Revenue=('Net_Sales', 'sum'),
    Avg_Transaction=('Net_Sales', 'mean'),
    Last_Purchase=('Order_Date', 'max')
).sort_values('Revenue', ascending=False)
customer_summary.head(10)

## Exercises
1. Identify the strongest and weakest month by revenue.
2. Find the product with the highest units sold, not just highest revenue.
3. Compare revenue and unique-customer rankings by region.
4. Calculate payment-method share using `value_counts(normalize=True)`.
5. Write five evidence-based business findings.